## Q1 — Greedy Best-First Search (GBFS)

In [1]:
import heapq

graph = {
    'A': [('B', 2), ('C', 3)],
    'B': [('D', 4), ('E', 5)],
    'C': [('F', 6)],
    'D': [('G', 7)],
    'E': [('G', 3)],
    'F': [('G', 8)],
    'G': []
}

heuristic = {
    'A': 6, 'B': 4, 'C': 5,
    'D': 2, 'E': 3, 'F': 6, 'G': 0
}

def greedy_bfs(graph, heuristic, start, goal):
    queue = [(heuristic[start], start, [start])]
    visited = set()
    expanded = []

    print(f"Greedy BFS: {start} to {goal}")
    print("-" * 40)

    while queue:
        h_val, node, path = heapq.heappop(queue)

        if node in visited:
            continue

        visited.add(node)
        expanded.append(node)
        print(f"  Expanding: {node}   h({node}) = {h_val}")

        if node == goal:
            return path, expanded

        for neighbor, cost in graph[node]:
            if neighbor not in visited:
                heapq.heappush(queue, (heuristic[neighbor], neighbor, path + [neighbor]))

    return None, expanded


path, expanded = greedy_bfs(graph, heuristic, 'A', 'G')

print()
print(f"Path found     : {' -> '.join(path)}")
print(f"Nodes expanded : {expanded}")

Greedy BFS: A to G
----------------------------------------
  Expanding: A   h(A) = 6
  Expanding: B   h(B) = 4
  Expanding: D   h(D) = 2
  Expanding: G   h(G) = 0

Path found     : A -> B -> D -> G
Nodes expanded : ['A', 'B', 'D', 'G']


---
## Q2 — A* Search Algorithm

In [2]:
import heapq

graph = {
    'A': [('B', 2), ('C', 3)],
    'B': [('D', 4), ('E', 5)],
    'C': [('F', 6)],
    'D': [('G', 7)],
    'E': [('G', 3)],
    'F': [('G', 8)],
    'G': []
}

heuristic = {
    'A': 6, 'B': 4, 'C': 5,
    'D': 2, 'E': 3, 'F': 6, 'G': 0
}

def astar(graph, heuristic, start, goal):
    queue = [(heuristic[start], 0, start, [start])]
    visited = {}
    expanded = []

    print(f"A* Search: {start} to {goal}")
    print("-" * 55)
    print(f"  {'Node':<6} {'g(n)':<8} {'h(n)':<8} {'f(n)':<8} Path")
    print("-" * 55)

    while queue:
        f, g, node, path = heapq.heappop(queue)

        if node in visited and visited[node] <= g:
            continue

        visited[node] = g
        expanded.append(node)
        print(f"  {node:<6} {g:<8} {heuristic[node]:<8} {f:<8} {' -> '.join(path)}")

        if node == goal:
            return path, g, expanded

        for neighbor, edge_cost in graph[node]:
            new_g = g + edge_cost
            new_f = new_g + heuristic[neighbor]
            if neighbor not in visited or visited[neighbor] > new_g:
                heapq.heappush(queue, (new_f, new_g, neighbor, path + [neighbor]))

    return None, float('inf'), expanded


path, cost, expanded = astar(graph, heuristic, 'A', 'G')

print()
print(f"Optimal path   : {' -> '.join(path)}")
print(f"Total cost     : {cost}")
print(f"Nodes expanded : {expanded}")

A* Search: A to G
-------------------------------------------------------
  Node   g(n)     h(n)     f(n)     Path
-------------------------------------------------------
  A      0        6        6        A
  B      2        4        6        A -> B
  C      3        5        8        A -> C
  D      6        2        8        A -> B -> D
  E      7        3        10       A -> B -> E
  G      10       0        10       A -> B -> E -> G

Optimal path   : A -> B -> E -> G
Total cost     : 10
Nodes expanded : ['A', 'B', 'C', 'D', 'E', 'G']


---
## Q3 — 8-Puzzle Heuristics

In [3]:
start_state = (1, 2, 5,
               3, 4, 0,
               6, 7, 8)

goal_state  = (1, 2, 3,
               4, 5, 6,
               7, 8, 0)

def print_puzzle(state):
    print("+---+---+---+")
    for row in range(3):
        tiles = [str(state[row*3+col]) if state[row*3+col] != 0 else ' ' for col in range(3)]
        print(f"| {' | '.join(tiles)} |")
        print("+---+---+---+")

print("Start:")
print_puzzle(start_state)
print("\nGoal:")
print_puzzle(goal_state)

Start:
+---+---+---+
| 1 | 2 | 5 |
+---+---+---+
| 3 | 4 |   |
+---+---+---+
| 6 | 7 | 8 |
+---+---+---+

Goal:
+---+---+---+
| 1 | 2 | 3 |
+---+---+---+
| 4 | 5 | 6 |
+---+---+---+
| 7 | 8 |   |
+---+---+---+


In [4]:
def misplaced_tiles(state, goal):
    count = 0
    for i in range(9):
        if state[i] != 0 and state[i] != goal[i]:
            count += 1
    return count


def manhattan_distance(state, goal):
    total = 0
    for i in range(9):
        tile = state[i]
        if tile == 0:
            continue
        current_row, current_col = i // 3, i % 3
        goal_index = goal.index(tile)
        goal_row, goal_col = goal_index // 3, goal_index % 3
        total += abs(current_row - goal_row) + abs(current_col - goal_col)
    return total


print(f"Misplaced Tiles    h(n) = {misplaced_tiles(start_state, goal_state)}")
print(f"Manhattan Distance h(n) = {manhattan_distance(start_state, goal_state)}")

Misplaced Tiles    h(n) = 6
Manhattan Distance h(n) = 11


In [5]:
import heapq

def get_neighbors(state):
    state = list(state)
    blank = state.index(0)
    row, col = blank // 3, blank % 3
    neighbors = []
    for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
        nr, nc = row + dr, col + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            new_state = state[:]
            swap = nr * 3 + nc
            new_state[blank], new_state[swap] = new_state[swap], new_state[blank]
            neighbors.append(tuple(new_state))
    return neighbors


def solve_puzzle(start, goal, h_func):
    queue = [(h_func(start, goal), 0, start, [])]
    visited = {}
    nodes_expanded = 0

    while queue:
        f, g, state, moves = heapq.heappop(queue)

        if state in visited and visited[state] <= g:
            continue
        visited[state] = g
        nodes_expanded += 1

        if state == goal:
            return moves, nodes_expanded

        for neighbor in get_neighbors(state):
            new_g = g + 1
            new_f = new_g + h_func(neighbor, goal)
            if neighbor not in visited or visited[neighbor] > new_g:
                heapq.heappush(queue, (new_f, new_g, neighbor, moves + [neighbor]))

    return None, nodes_expanded


moves1, expanded1 = solve_puzzle(start_state, goal_state, misplaced_tiles)
moves2, expanded2 = solve_puzzle(start_state, goal_state, manhattan_distance)

print(f"Misplaced Tiles    : {expanded1} nodes expanded, {len(moves1)} moves")
print(f"Manhattan Distance : {expanded2} nodes expanded, {len(moves2)} moves")
print()
if expanded2 <= expanded1:
    print("Manhattan Distance is more efficient — it expands fewer nodes.")
else:
    print("Misplaced Tiles performed better on this instance.")

Misplaced Tiles    : 5922 nodes expanded, 21 moves
Manhattan Distance : 862 nodes expanded, 21 moves

Manhattan Distance is more efficient — it expands fewer nodes.


---
## Q4 — Tic-Tac-Toe Board Representation & Utility

In [6]:
def make_board():
    return [' '] * 9


def print_board(board):
    print(f"\n {board[0]} | {board[1]} | {board[2]}")
    print("---+---+---")
    print(f" {board[3]} | {board[4]} | {board[5]}")
    print("---+---+---")
    print(f" {board[6]} | {board[7]} | {board[8]}\n")


def get_moves(board):
    return [i for i in range(9) if board[i] == ' ']


def check_winner(board):
    lines = [
        [0,1,2], [3,4,5], [6,7,8],
        [0,3,6], [1,4,7], [2,5,8],
        [0,4,8], [2,4,6]
    ]
    for a, b, c in lines:
        if board[a] == board[b] == board[c] and board[a] != ' ':
            return board[a]
    return None


def is_terminal(board):
    return check_winner(board) is not None or ' ' not in board


def utility(board):
    winner = check_winner(board)
    if winner == 'X': return 1
    if winner == 'O': return -1
    return 0


board1 = make_board()
print("Empty board:")
print_board(board1)
print(f"Legal moves  : {get_moves(board1)}")
print(f"Terminal?    : {is_terminal(board1)}")
print(f"Utility      : {utility(board1)}")

board2 = ['X','X','X','O','O',' ',' ',' ',' ']
print("\nX wins (top row):")
print_board(board2)
print(f"Winner       : {check_winner(board2)}")
print(f"Utility      : {utility(board2)}")

board3 = ['X','O','X','X','X','O','O','X','O']
print("\nDraw:")
print_board(board3)
print(f"Winner       : {check_winner(board3)}")
print(f"Utility      : {utility(board3)}")

Empty board:

   |   |  
---+---+---
   |   |  
---+---+---
   |   |  

Legal moves  : [0, 1, 2, 3, 4, 5, 6, 7, 8]
Terminal?    : False
Utility      : 0

X wins (top row):

 X | X | X
---+---+---
 O | O |  
---+---+---
   |   |  

Winner       : X
Utility      : 1

Draw:

 X | O | X
---+---+---
 X | X | O
---+---+---
 O | X | O

Winner       : None
Utility      : 0


---
## Q5 — Minimax Algorithm & Game Tree

In [7]:
class Node:
    def __init__(self, board, player, move=None):
        self.board    = board
        self.player   = player
        self.move     = move
        self.value    = None
        self.children = []

In [8]:
def minimax(node, is_max):
    if is_terminal(node.board):
        node.value = utility(node.board)
        return node.value

    if is_max:
        best = -999
        for move in get_moves(node.board):
            new_board = node.board[:]
            new_board[move] = 'X'
            child = Node(new_board, 'O', move)
            node.children.append(child)
            best = max(best, minimax(child, False))
        node.value = best
        return best
    else:
        best = 999
        for move in get_moves(node.board):
            new_board = node.board[:]
            new_board[move] = 'O'
            child = Node(new_board, 'X', move)
            node.children.append(child)
            best = min(best, minimax(child, True))
        node.value = best
        return best


def get_best_move(board):
    root = Node(board, 'X')
    best_move  = None
    best_value = -999

    print("Evaluating moves:")
    for move in get_moves(board):
        new_board = board[:]
        new_board[move] = 'X'
        child = Node(new_board, 'O', move)
        root.children.append(child)
        value = minimax(child, False)
        child.value = value
        print(f"  Index {move} -> value = {value}")
        if value > best_value:
            best_value = value
            best_move  = move

    root.value = best_value
    return best_move, best_value, root

In [9]:
board = ['X', 'X', ' ',
         'O', 'O', ' ',
         ' ', ' ', ' ']

print("Current board (X's turn):")
print_board(board)

best_move, best_value, root = get_best_move(board)

print(f"\nBest move for AI : index {best_move}")
print(f"Minimax value    : {best_value}")

board[best_move] = 'X'
print("\nBoard after AI plays:")
print_board(board)

winner = check_winner(board)
if winner:
    print(f"{winner} wins!")

Current board (X's turn):

 X | X |  
---+---+---
 O | O |  
---+---+---
   |   |  

Evaluating moves:
  Index 2 -> value = 1
  Index 5 -> value = 0
  Index 6 -> value = -1
  Index 7 -> value = -1
  Index 8 -> value = -1

Best move for AI : index 2
Minimax value    : 1

Board after AI plays:

 X | X | X
---+---+---
 O | O |  
---+---+---
   |   |  

X wins!


In [10]:
def show_tree(node, depth=0, max_depth=2):
    indent = "    " * depth
    if depth == 0:
        label = "ROOT (X to play)"
    else:
        played = 'X' if node.player == 'O' else 'O'
        label = f"{played} played at index {node.move}"
    print(f"{indent}[{label}]  value = {node.value}")
    if depth >= max_depth:
        if node.children:
            print(f"{indent}    ...")
        return
    for child in node.children:
        show_tree(child, depth + 1, max_depth)


print("Game Tree (first 2 levels):")
print("=" * 50)
show_tree(root)
print("=" * 50)
print("\n +1 = X wins   0 = draw   -1 = O wins")
print("The AI always picks the branch with the highest value.")

Game Tree (first 2 levels):
[ROOT (X to play)]  value = 1
    [X played at index 2]  value = 1
    [X played at index 5]  value = 0
        [O played at index 2]  value = 0
            ...
        [O played at index 6]  value = 1
            ...
        [O played at index 7]  value = 1
            ...
        [O played at index 8]  value = 1
            ...
    [X played at index 6]  value = -1
        [O played at index 2]  value = 0
            ...
        [O played at index 5]  value = -1
        [O played at index 7]  value = 1
            ...
        [O played at index 8]  value = 1
            ...
    [X played at index 7]  value = -1
        [O played at index 2]  value = -1
            ...
        [O played at index 5]  value = -1
        [O played at index 6]  value = 1
            ...
        [O played at index 8]  value = 1
            ...
    [X played at index 8]  value = -1
        [O played at index 2]  value = -1
            ...
        [O played at index 5]  value = -1

In [11]:
board2 = ['X', 'O', 'X',
          'X', 'O', ' ',
          ' ', ' ', ' ']

print("Scenario: O is about to win the middle column.")
print("The AI must BLOCK at index 7.")
print_board(board2)

move, value, _ = get_best_move(board2)
print(f"\nAI chose index : {move}")
print(f"Minimax value  : {value}")

board2[move] = 'X'
print("\nBoard after AI blocks:")
print_board(board2)

Scenario: O is about to win the middle column.
The AI must BLOCK at index 7.

 X | O | X
---+---+---
 X | O |  
---+---+---
   |   |  

Evaluating moves:
  Index 5 -> value = -1
  Index 6 -> value = 1
  Index 7 -> value = 0
  Index 8 -> value = -1

AI chose index : 6
Minimax value  : 1

Board after AI blocks:

 X | O | X
---+---+---
 X | O |  
---+---+---
 X |   |  

